# Desafio RAG #2 — Parent RAG

Usa `ParentDocumentRetriever`: **busca nos filhos, responde com os pais**.

## 1. Instalação

In [ ]:
!pip install -q "langchain<1.0" "langchain-community<1.0" langchain-openai langchain-chroma langchain-text-splitters pypdf chromadb sentence-transformers requests

## 2. Configuração e download do livro

In [ ]:
import os, getpass, requests

# Chave da OpenAI (digite quando pedir; não fica salva no notebook)
if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY: ")

# Baixa o PDF de "Os Sertões"
PDF_URL = "https://fundar.org.br/wp-content/uploads/2021/06/os-sertoes.pdf"
PDF_PATH = "os-sertoes.pdf"
if not os.path.exists(PDF_PATH):
    r = requests.get(PDF_URL, timeout=120)
    r.raise_for_status()
    with open(PDF_PATH, "wb") as f:
        f.write(r.content)
print("PDF ok:", os.path.getsize(PDF_PATH) // 1024, "KB")

## 3. Carregando o PDF

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

docs = PyPDFLoader(PDF_PATH).load()
print(f"{len(docs)} páginas carregadas")

## 4. LLM, embeddings e prompt

In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

prompt = ChatPromptTemplate.from_template("""Você é um especialista em "Os Sertões", de Euclides da Cunha.
Responda à pergunta usando APENAS o contexto abaixo. Se o contexto não for suficiente, diga que não encontrou a informação.
Responda em português, de forma clara e bem fundamentada.

Contexto:
{context}

Pergunta: {question}

Resposta:""")

def format_docs(docs):
    return "\n\n---\n\n".join(d.page_content for d in docs)

def build_chain(retriever):
    def run(question):
        retrieved = retriever.invoke(question)
        answer = (prompt | llm | StrOutputParser()).invoke(
            {"context": format_docs(retrieved), "question": question}
        )
        return answer, retrieved
    return run

## 5. Questões do desafio

In [ ]:
questions = [
    "Qual é a visão de Euclides da Cunha sobre o ambiente natural do sertão nordestino e como ele influencia a vida dos habitantes?",
    "Quais são as principais características da população sertaneja descritas por Euclides da Cunha? Como ele relaciona essas características com o ambiente em que vivem?",
    "Qual foi o contexto histórico e político que levou à Guerra de Canudos, segundo Euclides da Cunha?",
    "Como Euclides da Cunha descreve a figura de Antônio Conselheiro e seu papel na Guerra de Canudos?",
    "Quais são os principais aspectos da crítica social e política presentes em \"Os Sertões\"? Como esses aspectos refletem a visão do autor sobre o Brasil da época?",
]

## 6. Parent RAG: busca em chunks pequenos, resposta com o contexto grande

Indexa **chunks filhos** (pequenos, bons pra similaridade) mas devolve o **chunk pai** (maior) ao LLM. Assim a busca é precisa e o contexto enviado é rico.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore

parent_splitter = RecursiveCharacterTextSplitter(chunk_size=2500, chunk_overlap=250)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)

vectorstore = Chroma(collection_name="sertoes_parent", embedding_function=embeddings)
docstore = InMemoryStore()

retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=docstore,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
    search_kwargs={"k": 4},   # k filhos -> até k pais distintos
)
retriever.add_documents(docs)   # pode levar alguns minutos (muitos embeddings)
print("Filhos indexados:", vectorstore._collection.count())
print("Pais armazenados:", len(list(docstore.yield_keys())))

## Execução das questões

In [ ]:
ask = build_chain(retriever)

import json
results = {}
for i, q in enumerate(questions, 1):
    answer, retrieved = ask(q)
    results[q] = answer
    print(f"\n{'='*100}\nPERGUNTA {i}: {q}\n{'-'*100}\n{answer}")
    print(f"\n[{len(retrieved)} trechos recuperados | páginas: {sorted({d.metadata.get('page') for d in retrieved})}]")

with open("respostas_parent.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

### Observações

Compare com o Naive: o contexto maior melhorou as respostas sobre o contexto histórico (questão 3) e a crítica social (questão 5)? Qual o custo (tempo de indexação, tokens por consulta)?